# 专题：递归从入门到二叉树

适合阶段：刚学到二叉树，能看懂普通函数和 `if`，但还不理解递归怎样执行。

本专题只使用常见、直观、能逐行解释的 Python 写法。不会使用三元表达式、`lambda`、列表推导式或其他为了少写几行而增加理解成本的语法。

## 这份专题怎么学

建议分两次完成：

1. 第一次学到“二叉树的三种遍历”，重点理解递归的进入和返回。
2. 第二次学习最大深度、翻转二叉树、对称二叉树和最近公共祖先。
3. 所有带 `trace` 的代码都亲自运行，观察缩进怎样变化。
4. 暂时不要背整段答案，只背最后的“递归四步检查法”。

## 学完以后要达到什么程度

- 能用自己的话解释递归，不再把它理解成“函数莫名其妙调用自己”。
- 知道递归必须包含终止条件。
- 能看懂调用栈中“先进入，后返回”的过程。
- 能判断一段代码属于前序、中序还是后序处理。
- 能独立写出二叉树最大深度的递归解法。
- 面试时能说明递归的时间复杂度和空间复杂度。

## 1. 递归到底是什么

递归不是新的循环语法。它仍然只是普通函数，只不过这个函数在执行过程中又调用了自己。

真正重要的是：**每次调用时，问题都必须变小，并且最后要小到可以直接回答。**

例如计算 `1 + 2 + 3`：

- `sum_to(3)` 不直接算全部，它等于 `3 + sum_to(2)`。
- `sum_to(2)` 等于 `2 + sum_to(1)`。
- `sum_to(1)` 已经足够简单，直接返回 `1`。

这里的 `sum_to(1)` 就是递归的终止条件，也叫 **基本情况（Base Case）**。

In [ ]:
def countdown(number):
    if number == 0:
        print("结束")
        return

    print("现在是", number)
    countdown(number - 1)


countdown(3)

上面的执行顺序是：

```text
countdown(3)
  countdown(2)
    countdown(1)
      countdown(0) -> 结束
```

注意两件事：

- `if number == 0` 负责停下来。
- `number - 1` 让问题不断变小。

缺少其中任何一项，函数都可能一直调用自己，最后出现 `RecursionError`。

## 2. 最容易卡住的地方：进入和返回是两个方向

调用递归函数时，Python 会暂时停住当前这一层，先去完成下一层。

下一层完成以后，才会回到当前层，继续执行递归调用后面的代码。

下面故意在递归调用的前后各打印一次。

In [ ]:
def observe(number):
    if number == 0:
        print("到达最里面")
        return

    print("进入", number)
    observe(number - 1)
    print("返回", number)


observe(3)

运行结果中，“进入”是 `3、2、1`，“返回”却是 `1、2、3`。

可以把它想成进入三层房间：

```text
进入第 3 层 -> 进入第 2 层 -> 进入第 1 层
返回第 1 层 <- 返回第 2 层 <- 返回第 3 层
```

递归调用前面的代码按进入顺序执行，递归调用后面的代码按返回顺序执行。这个规律正是理解二叉树前序和后序处理的关键。

## 3. `return` 在递归中做什么

`return` 有两个作用：

1. 结束当前这一层函数。
2. 把当前这一层的结果交还给上一层。

只会调用自己但不接收返回值，是初学递归时最常见的错误之一。

In [ ]:
def sum_to(number):
    if number == 1:
        return 1

    smaller_answer = sum_to(number - 1)
    current_answer = number + smaller_answer
    return current_answer


answer = sum_to(4)
print(answer)  # 10

### 手算 `sum_to(4)`

先不断进入：

| 当前调用 | 暂时不能完成，因为还需要 |
|---|---|
| `sum_to(4)` | `sum_to(3)` |
| `sum_to(3)` | `sum_to(2)` |
| `sum_to(2)` | `sum_to(1)` |
| `sum_to(1)` | 直接返回 `1` |

再逐层返回：

| 返回到哪一层 | 这一层算出的结果 |
|---|---:|
| `sum_to(2)` | `2 + 1 = 3` |
| `sum_to(3)` | `3 + 3 = 6` |
| `sum_to(4)` | `4 + 6 = 10` |

不要试图一次在脑中展开十几层。先看当前层需要下一层提供什么，再相信下一层能完成自己的任务。

## 4. 写递归前必须回答三个问题

### 问题一：这个函数负责回答什么

例如：`max_depth(root)` 负责返回“以 `root` 为根的树的最大深度”。

### 问题二：最小问题是什么

例如：空树没有节点，所以深度是 `0`。

### 问题三：当前问题怎样使用更小问题的答案

例如：当前树的深度等于左右子树较大深度再加 `1`。

```text
当前树答案 = 组合(左子树答案, 右子树答案, 当前节点)
```

这三句话想不清楚时，先不要急着写代码。

## 5. 一个最普通的递归骨架

```python
def solve(当前问题):
    if 当前问题已经足够小:
        return 可以直接确定的答案

    更小问题的答案 = solve(更小的问题)
    当前问题的答案 = 根据更小问题的答案计算
    return 当前问题的答案
```

这不是要求你机械套模板，而是在提醒你检查：终止条件、问题变小、接住返回值、返回当前答案，一个都不能少。

## 6. 动手一：先写一个普通递归

请完成 `factorial(number)`，计算正整数的阶乘。

```text
4! = 4 × 3 × 2 × 1 = 24
1! = 1
```

先在纸上回答：函数负责什么、终止条件是什么、问题怎样变小。

In [ ]:
# 练习：计算正整数 number 的阶乘
# 输入：number 是大于等于 1 的整数。
# 输出：返回 number 的阶乘。
# 注意：必须写出终止条件，并让 number 每次减小。

def factorial_practice(number):
    # 在这里写你的代码
    pass

### 动手一参考答案

先自己写，再运行下面的答案。

In [ ]:
def factorial(number):
    if number == 1:
        return 1

    smaller_answer = factorial(number - 1)
    answer = number * smaller_answer
    return answer


assert factorial(1) == 1
assert factorial(4) == 24
print("阶乘测试通过")

## 7. 为什么二叉树特别适合递归

一棵二叉树由三部分组成：

1. 当前根节点。
2. 左子树。
3. 右子树。

左子树本身还是一棵二叉树，右子树本身也还是一棵二叉树。因此处理整棵树的方法，可以继续用于处理左右子树。

```text
        1
       / \
      2   3
     / \
    4   5
```

对于树题，最常见的更小问题就是：`root.left` 和 `root.right`。

## 8. 统一使用项目中已经学过的建树方法

下面的 `TreeNode` 和 `build_tree` 沿用 Day07 的写法：

- `TreeNode` 表示一个节点。
- `left` 指向左孩子，`right` 指向右孩子。
- `None` 表示这个位置没有节点。
- `build_tree` 按层序列表创建树。

建树使用队列只是为了准备测试数据，不是本节递归算法的一部分。

In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None

    root = TreeNode(values[0])
    queue = deque([root])
    index = 1

    while queue and index < len(values):
        node = queue.popleft()

        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1

        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1

    return root


root = build_tree([1, 2, 3, 4, 5])
print(root.val)       # 1
print(root.left.val)  # 2

## 9. 树递归最常见的终止条件

```python
if root is None:
    return ...
```

为什么总是在判断 `None`？

因为从叶子节点继续访问 `left` 或 `right`，最终一定会走到空位置。空位置就是树递归中最自然的最小问题。

返回什么不能死记：

- 统计节点数量时，空树贡献 `0`。
- 计算最大深度时，空树深度是 `0`。
- 收集遍历结果时，空树不添加任何内容。
- 判断某个条件时，返回 `True` 还是 `False` 要根据题意决定。

## 10. 前序、中序、后序：区别只是处理当前节点的位置

对于当前节点，可以做三件事：

1. 处理当前节点。
2. 递归处理左子树。
3. 递归处理右子树。

| 遍历方式 | 顺序 | 常见用途 |
|---|---|---|
| 前序遍历 | 当前、左、右 | 复制树、记录路径、从根向下传信息 |
| 中序遍历 | 左、当前、右 | 二叉搜索树得到有序序列 |
| 后序遍历 | 左、右、当前 | 汇总子树答案，如深度、节点数量 |

“前、中、后”说的是当前节点放在左右子树的什么位置。

In [ ]:
def preorder_dfs(node, result):
    if node is None:
        return

    result.append(node.val)
    preorder_dfs(node.left, result)
    preorder_dfs(node.right, result)


def preorder_traversal(root):
    result = []
    preorder_dfs(root, result)
    return result


def inorder_dfs(node, result):
    if node is None:
        return

    inorder_dfs(node.left, result)
    result.append(node.val)
    inorder_dfs(node.right, result)


def inorder_traversal(root):
    result = []
    inorder_dfs(root, result)
    return result


def postorder_dfs(node, result):
    if node is None:
        return

    postorder_dfs(node.left, result)
    postorder_dfs(node.right, result)
    result.append(node.val)


def postorder_traversal(root):
    result = []
    postorder_dfs(root, result)
    return result


tree = build_tree([1, 2, 3, 4, 5])
print("前序：", preorder_traversal(tree))
print("中序：", inorder_traversal(tree))
print("后序：", postorder_traversal(tree))

这里没有使用列表拼接，而是把同一个 `result` 列表传给每一层：

- `result.append(node.val)` 写在两个递归调用之前，就是前序。
- 写在左右递归之间，就是中序。
- 写在两个递归调用之后，就是后序。

`preorder_dfs` 负责递归访问节点，`preorder_traversal` 负责创建并返回结果列表。分成两个函数虽然多写几行，但职责清楚，也方便面试时逐行解释。

## 11. 用打印看懂树递归的进入和返回

`depth` 只用于控制打印缩进，让你看见当前递归到了第几层。

In [ ]:
def preorder_trace(node, depth):
    if node is None:
        print("  " * depth, "遇到 None，返回")
        return

    print("  " * depth, "进入节点", node.val)
    preorder_trace(node.left, depth + 1)
    preorder_trace(node.right, depth + 1)
    print("  " * depth, "离开节点", node.val)


trace_tree = build_tree([1, 2, 3])
preorder_trace(trace_tree, 0)

## 12. 高频题：二叉树的最大深度

**专业名称：深度优先搜索 + 后序递归（Depth-First Search + Postorder Recursion）**

函数定义：`max_depth(root)` 返回以 `root` 为根的树的最大深度。

拆解：

1. 空树的深度是 `0`。
2. 让递归分别计算左子树深度和右子树深度。
3. 当前节点的深度是较大子树深度加 `1`。

```text
当前深度 = max(左子树深度, 右子树深度) + 1
```

之所以属于后序思路，是因为必须先得到左右子树答案，才能计算当前节点答案。

In [ ]:
# 题目：二叉树的最大深度
# 解法：深度优先搜索 + 后序递归（Depth-First Search + Postorder Recursion）
# 输入：root 是二叉树根节点。
# 目标：计算从根节点到最远叶子节点经过的节点数量。
# 输出：返回最大深度；空树返回 0。
# 注意：当前节点的深度需要使用左右子树返回的深度。

def max_depth_practice(root):
    # 在这里写你的代码
    pass

### 最大深度参考答案

In [ ]:
def max_depth(root):
    if root is None:
        return 0

    left_depth = max_depth(root.left)
    right_depth = max_depth(root.right)
    current_depth = max(left_depth, right_depth) + 1
    return current_depth


assert max_depth(build_tree([])) == 0
assert max_depth(build_tree([1])) == 1
assert max_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 3
print("最大深度测试通过")

## 13. 高频题：翻转二叉树

**专业名称：深度优先搜索 + 递归交换（Depth-First Search + Recursive Swap）**

不要使用一行交换来缩短代码。为了清楚看到步骤，我们显式保存左右子树：

1. 空节点直接返回 `None`。
2. 递归翻转原来的左子树。
3. 递归翻转原来的右子树。
4. 把两个结果交换后接回当前节点。

In [ ]:
def invert_tree(root):
    if root is None:
        return None

    inverted_left = invert_tree(root.left)
    inverted_right = invert_tree(root.right)

    root.left = inverted_right
    root.right = inverted_left
    return root


tree = build_tree([4, 2, 7, 1, 3, 6, 9])
inverted_tree = invert_tree(tree)
assert preorder_traversal(inverted_tree) == [4, 7, 9, 6, 2, 3, 1]
print("翻转二叉树测试通过")

## 14. 进阶一点：为什么对称二叉树要同时递归两个节点

**专业名称：成对递归 / 镜像递归（Paired Recursion / Mirror Recursion）**

判断镜像时，不是只看一个节点，而是每次比较一对节点 `left` 和 `right`：

- 两个都是 `None`：这一对对称。
- 只有一个是 `None`：不对称。
- 值不同：不对称。
- 值相同：继续比较外侧一对和内侧一对。

外侧是 `left.left` 对 `right.right`，内侧是 `left.right` 对 `right.left`。

In [ ]:
def is_mirror(left, right):
    if left is None and right is None:
        return True

    if left is None or right is None:
        return False

    if left.val != right.val:
        return False

    outside_is_mirror = is_mirror(left.left, right.right)
    inside_is_mirror = is_mirror(left.right, right.left)
    return outside_is_mirror and inside_is_mirror


def is_symmetric(root):
    if root is None:
        return True

    return is_mirror(root.left, root.right)


assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])) is True
assert is_symmetric(build_tree([1, 2, 2, None, 3, None, 3])) is False
print("对称二叉树测试通过")

## 15. 再进阶：最近公共祖先为什么是后序递归

**专业名称：后序深度优先搜索（Postorder Depth-First Search）**

函数向左右子树询问：你那里有没有找到 `p` 或 `q`？

- 左右都找到了：说明 `p` 和 `q` 分别位于两边，当前节点就是最近公共祖先。
- 只有左边找到了：把左边找到的节点继续向上返回。
- 只有右边找到了：把右边找到的节点继续向上返回。
- 都没找到：返回 `None`。

这道题第一次看不懂很正常。现阶段重点是看清“左右子树先返回信息，当前节点再汇总”，不要求马上默写。

In [ ]:
def lowest_common_ancestor(root, p, q):
    if root is None:
        return None

    if root is p or root is q:
        return root

    left_result = lowest_common_ancestor(root.left, p, q)
    right_result = lowest_common_ancestor(root.right, p, q)

    if left_result is not None and right_result is not None:
        return root

    if left_result is not None:
        return left_result

    return right_result


lca_tree = build_tree([3, 5, 1, 6, 2, 0, 8])
p = lca_tree.left
q = lca_tree.right
answer = lowest_common_ancestor(lca_tree, p, q)
assert answer is lca_tree
print("最近公共祖先测试通过，答案是", answer.val)

## 16. 递归、回溯和循环不要混在一起

### 递归

一种函数调用方式：函数通过更小的同类问题解决当前问题。

### 回溯

回溯通常使用递归实现，但还包含“做选择、继续搜索、撤销选择”。排列、组合、子集常用回溯。

### 循环

循环使用 `for` 或 `while` 重复执行代码，通常不会产生一层层函数调用。

递归不一定比循环高级。数组题通常更适合循环，二叉树天然由左右子树组成，所以经常更适合递归。

## 17. 递归的时间复杂度和空间复杂度

以二叉树最大深度为例：

### 时间复杂度：`O(n)`

树中的每个节点都会被访问一次，`n` 是节点数量。

### 空间复杂度：`O(h)`

递归调用栈最多同时保存从根到当前节点的一条路径，`h` 是树的高度。

- 平衡二叉树：高度大约是 `log n`，栈空间约为 `O(log n)`。
- 极度倾斜的树：高度可能是 `n`，最坏栈空间是 `O(n)`。

面试时不要简单说“递归空间一定是 `O(n)`”。更准确的说法是 `O(h)`，然后说明最坏情况下 `h = n`。

## 18. 五个最常见的错误

### 错误一：没有终止条件

函数会不断调用自己，最后出现 `RecursionError`。

### 错误二：问题没有变小

例如一直调用 `solve(number)`，而不是 `solve(number - 1)`。

### 错误三：忘记接住或返回递归结果

```python
max_depth(root.left)  # 算了，但没有保存结果
```

应该保存为 `left_depth`，并参与当前答案计算。

### 错误四：混淆 `print` 和 `return`

`print` 只是给人看，`return` 才会把结果交给上一层。

### 错误五：一开始就追踪整棵大树

先用 1 到 3 个节点的小树调试。树太大时，打印结果会迅速变乱。

## 19. 不理解时怎样调试递归

按照这个顺序检查：

1. 把输入缩小到 1 到 3 个节点。
2. 在函数第一行打印“进入哪个节点”。
3. 在每个 `return` 前打印“返回什么”。
4. 使用 `depth` 增加缩进，看清递归层数。
5. 先单独检查终止条件，再检查左右递归，最后检查答案组合。

不要一上来用完整测试树，也不要同时改很多行。一次只验证一个判断。

In [ ]:
def max_depth_trace(root, depth):
    if root is None:
        print("  " * depth, "None 返回 0")
        return 0

    print("  " * depth, "进入节点", root.val)
    left_depth = max_depth_trace(root.left, depth + 1)
    right_depth = max_depth_trace(root.right, depth + 1)
    answer = max(left_depth, right_depth) + 1
    print("  " * depth, "节点", root.val, "返回", answer)
    return answer


small_tree = build_tree([1, 2, 3])
max_depth_trace(small_tree, 0)

## 20. 面试时怎样表达

以最大深度为例，可以这样说：

> 我使用深度优先搜索的后序递归。函数 `max_depth(root)` 返回以当前节点为根的子树最大深度。空节点返回 0；非空节点分别递归计算左右子树深度，取较大值再加 1。每个节点访问一次，所以时间复杂度是 `O(n)`；递归栈深度等于树高，所以空间复杂度是 `O(h)`，最坏为 `O(n)`。

这段表达包含了面试官最关心的五件事：专业名称、函数定义、终止条件、递推关系、复杂度。

## 21. 递归四步检查法

以后看到树的递归题，按下面四步写：

1. **定职责**：函数到底返回什么，或者负责做什么。
2. **写出口**：当前节点是 `None` 时返回什么。
3. **找小问题**：通常是递归处理 `root.left` 和 `root.right`。
4. **组答案**：当前节点怎样利用左右子树结果得到自己的结果。

简写成一句话：

```text
定职责 -> 写出口 -> 递归左右 -> 组合返回
```

如果是遍历型题目，“组合返回”可能变成“在正确的位置处理当前节点”。

## 22. 今日自测

不看前文，尝试回答：

1. 为什么递归必须有终止条件？
2. `print` 和 `return` 在递归中有什么区别？
3. 为什么最大深度适合后序递归？
4. 前序、中序、后序中的“前、中、后”指什么？
5. 二叉树递归的空间复杂度为什么常写成 `O(h)`？
6. 对称二叉树为什么每次需要同时比较两个节点？

有两道答不上来，就回到对应章节重新运行小例子，不要急着背题解。

## 23. 最后总结

- 递归就是普通函数通过更小的同类问题解决当前问题。
- 终止条件负责停下来，参数变化负责让问题逐渐接近终止条件。
- 递归先进入、后返回；调用后的代码会在返回阶段执行。
- 树由根、左子树、右子树组成，因此天然适合递归。
- 遍历题关注“何时处理当前节点”，求值题关注“左右子树返回什么”。
- 先写清函数职责，再写代码，比背模板更可靠。

最低掌握目标：能不看答案写出 `max_depth`，并逐行解释每个 `return` 的作用。